In [0]:
%run ./01_setup_paths_and_schema

In [0]:
dbutils.fs.rm(
    f"{bronze_path}/patients",
    True
)

In [0]:
from pyspark.sql.functions import *

# Header validation only
header_df = spark.read.format("csv") \
.option("header",True) \
.load(f"{source_path}/patients")

raw_columns = header_df.columns

expected_columns = [

    field.name

    for field in patients_schema.fields

]

missing_columns = list(

    set(expected_columns)

    - set(raw_columns)

)

if len(missing_columns) > 0:

    raise Exception(

        f"Missing Columns : {missing_columns}"

    )

# Read again USING EXPLICIT SCHEMA

df = spark.read.format("csv") \
.option("header",True) \
.schema(patients_schema) \
.load(f"{source_path}/patients")

df = df.withColumn(

    "ingestion_time",

    current_timestamp()

)

df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.save(f"{bronze_path}/patients")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

raw_df = spark.read.format("csv") \
.option("header",True) \
.load(f"{source_path}/patients")

raw_columns = raw_df.columns

expected_columns = [

    field.name

    for field in patients_schema.fields

]

missing_columns = list(

    set(expected_columns)

    - set(raw_columns)

)

print("Missing =", missing_columns)

# ----------------------------------
# ADD MISSING COLUMNS
# ----------------------------------

for field in patients_schema.fields:

    if field.name not in raw_df.columns:

        raw_df = raw_df.withColumn(

            field.name,

            lit(None).cast(field.dataType)

        )

# reorder columns

raw_df = raw_df.select(

    *expected_columns

)

# apply metadata cols

bronze_df = raw_df.withColumn(

    "ingestion_time",

    current_timestamp()

)

bronze_df.write \
.format("delta") \
.mode("overwrite") \
.save(f"{bronze_path}/patients")
display(bronze_df)

In [0]:


from pyspark.sql.functions import *

df = spark.read.format("csv") \
.option("header",True) \
.schema(patients_schema) \
.load(f"{source_path}/patients")

df = df.withColumn(

    "ingestion_time",

    current_timestamp()

)

df.write \
.format("delta") \
.mode("overwrite") \
.save(f"{bronze_path}/patients")